# Predict reported transaction fraud on Databricks

Which transactions will receive a simulated confirmed-fraud report within seven days? Train Kumo Engine on four related synthetic tables, then score newly added transactions.

Use a Databricks GPU notebook with local SSD scratch and a Unity Catalog volume for retained files. Follow [Databricks setup](https://docs.nvidia.com/sdgm/kumo-engine/get-started-with-kumo-engine/installation/databricks-notebook). The example writes local synthetic files; you load them into new tables in your own catalog and schema at the two indicated steps.


## Install Kumo Engine

Install the Databricks integration into this notebook kernel before restarting Python.


In [ ]:
%pip install "kumo-engine[databricks]"

<br>

## Restart Python

Restart before defining configuration or importing the installed package.


In [ ]:
dbutils.library.restartPython()


<br>

## Set the example parameters

Fill in your workspace, warehouse, catalog, schema, volume, and secret references. Each run uses new source/output table names; keep existing warehouse tables unchanged. The 0.5 classification threshold is illustrative.


In [ ]:
from pathlib import Path

DATABRICKS_HOST = ""
WAREHOUSE_ID = ""
CATALOG = ""
SCHEMA = ""
CLIENT_ID = ""
SECRET_SCOPE = ""
CLIENT_SECRET_KEY = ""
ARTIFACT_ROOT = Path("/Volumes/<catalog>/<schema>/<volume>/runs")
SCRATCH_ROOT = Path("/local_disk0/kumo-scratch")
SEED = 42
N_HISTORY = 8000
N_NEW = 100
HISTORY_START = "2019-01-01"
HISTORY_DAYS = 600
SCORING_START = "2020-09-01"
TRAIN_START = "2019-02-01 00:00:00"
TRAIN_END = "2020-05-31 00:00:00"
THRESHOLD = 0.5
REMOVE_RUN_SCRATCH = False

Import the workflow tools and create separate, uniquely named scratch and retained run directories. The printed mapping will identify this run's warehouse tables.

In [ ]:
import shutil
import tempfile
from uuid import uuid4

import numpy as np
import pandas as pd
from IPython.display import display
import kumo.engine as kumo
from kumoapi.graph import Edge
from kumoapi.typing import Stype
from kumoapi.model_plan import RunMode, TrainingTableGenerationPlan
from kumo.engine.trainer.output_config import TrainingTableOutputConfig
from kumoapi.jobs import MetadataField, PredictionArtifactType, WriteMode

run_id = uuid4().hex[:12]
TABLE_PREFIX = f"kumo_fraud_{run_id}_"
run_dir = Path(tempfile.mkdtemp(prefix=f"kumo-fraud-{run_id}-"))
scratch_dir = SCRATCH_ROOT / run_id
scratch_dir.mkdir(parents=True, exist_ok=False)
RUN_ROOT = ARTIFACT_ROOT / run_id
RUN_ROOT.mkdir(parents=True, exist_ok=False)
kumo.configure(scratch_dir=str(scratch_dir))
print("Local data:", run_dir)
print("Retained results:", RUN_ROOT)
print("Warehouse table prefix:", TABLE_PREFIX)


<br>

## Generate synthetic example data

Run the next cell to create accounts, merchants, historical transactions, and reports, plus a separate transaction batch reserved for scoring. You can skim the generation code; the connector workflow starts at **Load the historical tables**.
Merchant risk adds noisy signal. Historical confirmed reports arrive two days after a transaction; reserved transactions have no supplied report outcomes.

In [ ]:
rng = np.random.default_rng(SEED)
customers = pd.DataFrame({
    "cc_num": [f"synthetic-card-{i}" for i in range(300)],
    "age": rng.integers(18, 80, 300),
})
merchants = pd.DataFrame({
    "merchant": [f"synthetic-merchant-{i}" for i in range(50)],
    "risk": rng.uniform(0, 1, 50),
})

def make_transactions(count, prefix, start, days):
    merchant_ids = rng.integers(0, len(merchants), count)
    transactions = pd.DataFrame({
        "trans_num": [f"synthetic-{prefix}-transaction-{i}" for i in range(count)],
        "cc_num": rng.choice(customers.cc_num, count),
        "merchant": merchants.merchant.to_numpy()[merchant_ids],
        "amt": rng.lognormal(4, 1, count),
        "trans_date_trans_time": pd.Timestamp(start) + pd.to_timedelta(
            rng.integers(0, days * 86400, count), unit="s"
        ),
    })
    flagged = rng.random(count) < 0.02 + 0.4 * merchants.risk.to_numpy()[merchant_ids]
    reports = transactions.loc[flagged, ["trans_num", "trans_date_trans_time"]].rename(
        columns={"trans_date_trans_time": "report_time"}
    )
    reports["report_time"] += pd.Timedelta(days=2)
    reports["is_real_fraud"] = 1
    return transactions, reports

historical_transactions, reports = make_transactions(N_HISTORY, "history", HISTORY_START, HISTORY_DAYS)
new_transactions, _ = make_transactions(N_NEW, "new", SCORING_START, 1)
assert N_NEW >= 2 and new_transactions.trans_num.is_unique
assert set(new_transactions.trans_num).isdisjoint(historical_transactions.trans_num)
assert new_transactions.cc_num.isin(customers.cc_num).all()
assert new_transactions.merchant.isin(merchants.merchant).all()
assert new_transactions.dtypes.equals(historical_transactions.dtypes)
assert historical_transactions.trans_date_trans_time.max() < new_transactions.trans_date_trans_time.min()
frames = {"customers": customers, "merchants": merchants,
          "transactions": historical_transactions, "fraud_reports": reports}
history_dir = run_dir / "history"
history_dir.mkdir()
for name, frame in frames.items():
    frame.to_parquet(history_dir / f"{TABLE_PREFIX}{name}.parquet", index=False)
display(pd.DataFrame({"table": list(frames), "rows": [len(frame) for frame in frames.values()]}))
print("Historical files:", history_dir)

<br>

## Load the historical tables

Load only the four files in `history_dir` into new Unity Catalog tables in the configured catalog and schema, using each filename stem as its table name. Preserve key types, column names, and timestamps. The notebook does not upload or overwrite source tables. Keep the reserved `new_transactions` out of these tables until **Batch predictions**.

After loading, connect with the service principal whose secret is stored in your configured Databricks secret scope.

In [ ]:
identity = kumo.DatabricksIdentity(
    host=DATABRICKS_HOST, client_id=CLIENT_ID,
    client_secret=dbutils.secrets.get(SECRET_SCOPE, CLIENT_SECRET_KEY),
)
connector = kumo.DatabricksConnector(
    identity=identity, warehouse_id=WAREHOUSE_ID, catalog=CATALOG, schema_name=SCHEMA,
)


<br>

## Check source access

Inspect the four newly loaded tables before defining their relationships.


In [ ]:
for name in frames:
    print(TABLE_PREFIX + name, [column.name for column in connector[TABLE_PREFIX + name].columns])

<br>

## Build the fraud graph

Connect transactions to accounts, merchants, and timestamped reports using [Engine tables and relationships](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/set-up-a-graph/graphs). Explicit graph names keep the predictive query independent of this run's warehouse table prefix.


In [ ]:
customers_table = kumo.Table.from_source_table(connector[TABLE_PREFIX + "customers"], primary_key="cc_num")
merchants_table = kumo.Table.from_source_table(connector[TABLE_PREFIX + "merchants"], primary_key="merchant")
transactions_table = (
    kumo.Table.from_source_table(connector[TABLE_PREFIX + "transactions"],
        primary_key="trans_num", time_column="trans_date_trans_time")
    .with_column("cc_num", stype=Stype.ID).with_column("merchant", stype=Stype.ID)
)
reports_table = (
    kumo.Table.from_source_table(connector[TABLE_PREFIX + "fraud_reports"], time_column="report_time")
    .with_column("trans_num", stype=Stype.ID)
)
graph = kumo.Graph(
    tables={"customers": customers_table, "merchants": merchants_table,
            "transactions": transactions_table, "fraud_reports": reports_table},
    edges=[Edge("transactions", "cc_num", "customers"),
           Edge("transactions", "merchant", "merchants"),
           Edge("fraud_reports", "trans_num", "transactions")],
)
graph.validate()

<br>

## Inspect the graph

Inspect the actual graph before training. [Graph visualization](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/set-up-a-graph/graphs#visualize-table-relationships) is a commented preview because the supported package does not expose it; the relationship summary is executable.

In [ ]:
# Preview only: graph.visualize(show_cols=True)
display(pd.DataFrame([{"source": edge.src_table, "foreign_key": edge.fkey,
                       "destination": edge.dst_table} for edge in graph.edges]))

Materialize the historical graph once; this training snapshot stays unchanged when new transactions are loaded later.

In [ ]:
snapshot = graph.snapshot()
print("Snapshot:", snapshot.id)

<br>

## Define the prediction question

Predict whether a transaction receives any confirmed report during the next seven days. The temporal window makes report arrival part of the outcome, rather than an outcome available at transaction time. Validate the [predictive query](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/predictive-queries) before creating training rows.


In [ ]:
QUERY = """
PREDICT COUNT(fraud_reports.* WHERE fraud_reports.is_real_fraud = 1, 0, 7, days) >= 1
FOR EACH transactions.trans_num
""".strip()

pquery = kumo.PredictiveQuery(graph=graph, query=QUERY)
pquery.validate()
print("Task:", pquery.get_task_type())


<br>

## Prepare historical training rows

Use historical anchors from the configured interval and chronological 70/15/15 splits, so validation and test follow training. The source history extends beyond the last anchor to cover the seven-day outcome window.


In [ ]:
generation_plan = TrainingTableGenerationPlan(
    start_time=TRAIN_START, end_time=TRAIN_END,
    split="TemporalSplit([0.7, 0.15, 0.15])",
)
train_table = pquery.generate_training_table(snapshot, plan=generation_plan)
train_table.export(TrainingTableOutputConfig(
    connector=kumo.FileConnector(root_dir=str(run_dir)), table_name="training_rows/"
))
training_rows = pd.read_parquet(run_dir / "training_rows")
display(training_rows.head())
display(training_rows.groupby("SPLIT").agg(
    rows=("ENTITY1", "size"), first_anchor=("TIMESTAMP", "min"), last_anchor=("TIMESTAMP", "max")
))
assert set(training_rows.SPLIT) == {0, 1, 2}


<br>

## Train the model

Use the supported FAST preset, select by AUPRC for the report-positive class, and disable refitting to keep validation and test populations separate.

In [ ]:
plan = pquery.suggest_model_plan(snapshot, run_mode=RunMode.FAST)
plan.training_job.refit_trainval = False
plan.training_job.refit_full = False
plan.training_job.metrics = ["auprc", "auroc"]
plan.training_job.tune_metric = "auprc"
trainer = kumo.Trainer(plan)
fit_result = trainer.fit(train_table, snapshot=snapshot)

<br>

## Evaluate

AUPRC summarizes retrieval of the report-positive class; AUROC measures ranking across thresholds. These fit-reported synthetic metrics are **demonstration only**. Select settings using validation and reserve test results for final assessment.

In [ ]:
print("Synthetic-data metrics — demonstration only")
display(pd.Series(fit_result.metrics, name="value").rename_axis("metric").to_frame())

<br>

## Batch predictions

Prepare the reserved transactions for addition to this run's source table, plus a label-free population containing their IDs and event-time anchors. These IDs did not exist in the training snapshot.


In [ ]:
append_dir = run_dir / "append"
append_dir.mkdir(exist_ok=True)
new_transactions.to_parquet(append_dir / f"{TABLE_PREFIX}transactions.parquet", index=False)
scoring_population = new_transactions[["trans_num", "trans_date_trans_time"]].rename(
    columns={"trans_num": "ENTITY1", "trans_date_trans_time": "TIMESTAMP"}
)
assert set(scoring_population.columns) == {"ENTITY1", "TIMESTAMP"}
assert not scoring_population.duplicated(["ENTITY1", "TIMESTAMP"]).any()
scoring_population.to_parquet(append_dir / f"{TABLE_PREFIX}prediction_population.parquet", index=False)
print("Append to", TABLE_PREFIX + "transactions:", append_dir / f"{TABLE_PREFIX}transactions.parquet")
print("Create population table:", TABLE_PREFIX + "prediction_population")
display(scoring_population.head())

<br>

### Load the new records and refresh the graph

Append `append_dir`'s transaction file to this run's existing transaction table **once**, without replacing historical rows. Load its population file into the new table named above. Do not load simulated future reports for the new transactions.

After loading, materialize a fresh snapshot. The accepted population check verifies that every reserved ID and timestamp is available in that snapshot; stop and check the upload if it fails.

In [ ]:
scoring_snapshot = graph.snapshot()
assert scoring_snapshot.id != snapshot.id, "Append the reserved transactions before refreshing the graph."
prediction_table = kumo.PredictionTable.from_source_table(
    connector[TABLE_PREFIX + "prediction_population"], pq=pquery, snapshot=scoring_snapshot,
    plan=trainer.training_table_generation_plan,
)
population = pd.read_parquet(prediction_table.table_path)
assert not any(column.startswith("TARGET") for column in population)
population["TIMESTAMP"] = pd.to_datetime(population.TIMESTAMP)
pd.testing.assert_frame_equal(
    population[["ENTITY1", "TIMESTAMP"]].sort_values("ENTITY1").reset_index(drop=True),
    scoring_population.sort_values("ENTITY1").reset_index(drop=True), check_dtype=False,
)
PREDICTION_TABLE = SCHEMA + "__kumo__" + TABLE_PREFIX + "predictions_" + uuid4().hex[:8]

<br>

### Score the new transactions

Use the trained model with the refreshed snapshot and save results to a uniquely named warehouse table. Include anchor timestamps so each score can be matched to its request.

In [ ]:
pred_result = trainer.predict(
    prediction_table, snapshot=scoring_snapshot,
    output_config=kumo.PredictionOutputConfig(
        connector=connector, artifact_type=PredictionArtifactType.PREDICTIONS,
        table_name=PREDICTION_TABLE, write_mode=WriteMode.OVERWRITE,
        extra_fields=[MetadataField.ANCHOR_TIMESTAMP],
    ),
    num_workers=1, binary_classification_threshold=THRESHOLD,
)


<br>

### Inspect and save the results

Expect one output per new transaction. `True_PROB` is the probability of a confirmed report within seven days; `TARGET_PRED` applies the illustrative threshold. Verify the ID and anchor together, inspect a short sample, and retain a Parquet copy.


In [ ]:
scores = pred_result.frame
scores["ANCHOR_TIMESTAMP"] = pd.to_datetime(scores.ANCHOR_TIMESTAMP)
expected = scoring_population.rename(columns={"ENTITY1": "ENTITY", "TIMESTAMP": "ANCHOR_TIMESTAMP"})
alignment = expected.merge(scores, on=["ENTITY", "ANCHOR_TIMESTAMP"], how="outer", validate="one_to_one", indicator=True)
assert alignment._merge.eq("both").all()
probabilities = scores[["False_PROB", "True_PROB"]].to_numpy()
assert np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all()
assert np.allclose(probabilities.sum(axis=1), 1, atol=1e-6)
scores.to_parquet(RUN_ROOT / "predictions.parquet", index=False)
display(scores[["ENTITY", "ANCHOR_TIMESTAMP", "True_PROB", "TARGET_PRED"]].head(10))
print("Saved:", RUN_ROOT / "predictions.parquet")
print("Warehouse output:", PREDICTION_TABLE)

<br>

## Retain and reuse

Copy the complete model bundle to the configured volume, then reload it without fitting. Retain the source tables and scoring population separately. A new process must recreate the connector, graph, and snapshot before prediction; see [publish and retrieve models](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/publish-and-retrieve-models).


In [ ]:
model_path = RUN_ROOT / "model"
shutil.copytree(Path(fit_result.bundle_dir), model_path)
reloaded_trainer = kumo.Trainer.load(model_path)
print("Retained model:", model_path)


<br>

## Cleanup

Keep the retained model, predictions, and warehouse tables for inspection. Enable `REMOVE_RUN_SCRATCH` to remove only this run's scratch; rebuild snapshots before scoring again. The notebook leaves source and output warehouse tables intact.

In [ ]:
if REMOVE_RUN_SCRATCH:
    shutil.rmtree(scratch_dir)
print("Local source files:", run_dir)
print("Retained artifacts:", RUN_ROOT)